**Tier-Based Access in HAB**

In [ ]:
# tier_config.py ->  feature restrictions based on a user's subscription level

TIER_LIMITS = {
    "free": {
        "modalities": ["chlor_a"],
        "days": 5,
        "threads": 1,
        "max_calls": 3,
    },
    "tier1": {
        "modalities": ["chlor_a", "Rrs_412", "Rrs_443"],
        "days": 10,
        "threads": 4,
        "max_calls": 30,
    },
    "tier2": {
        "modalities": ["chlor_a", "Rrs_412", "Rrs_443"],
        "days": 10,
        "threads": 10,
        "max_calls": 1000,
    }
}


In [ ]:
# auth.py -> identifying users and tracking their API usage based on their API keys.

USER_DB = {
    "api-key-free": {"tier": "free", "usage_count": 1},
    "api-key-tier1": {"tier": "tier1", "usage_count": 5},
    "api-key-tier2": {"tier": "tier2", "usage_count": 12},
}

def get_user_by_api_key(api_key: str):
    return USER_DB.get(api_key)

def increment_usage(api_key: str):
    if api_key in USER_DB:
        USER_DB[api_key]["usage_count"] += 1


In [ ]:
# validator.py -> authenticates the user via API key and enforces tier-based limits on modalities, date range, and API usage before allowing further processing.

from fastapi import HTTPException
from tier_config import TIER_LIMITS
from auth import get_user_by_api_key, increment_usage

def validate_user_request(api_key, modalities, days):
    user = get_user_by_api_key(api_key)
    if not user:
        raise HTTPException(status_code=403, detail="Invalid API key.")

    tier = user["tier"]
    limits = TIER_LIMITS[tier]

    if len(modalities) > len(limits["modalities"]):
        raise HTTPException(403, detail="Your tier supports fewer modalities.")

    if any(m not in limits["modalities"] for m in modalities):
        raise HTTPException(403, detail="You are using unsupported modality.")

    if days > limits["days"]:
        raise HTTPException(403, detail="Requested days exceed your plan.")

    if user["usage_count"] >= limits["max_calls"]:
        raise HTTPException(429, detail="Monthly API call limit exceeded.")

    increment_usage(api_key)
    return tier, limits


In [ ]:
# main.py (FastAPI)

from fastapi import FastAPI, Form
from validator import validate_user_request
from your_module import generate_prediction_datacube

app = FastAPI()

@app.post("/predict")
async def predict(
    api_key: str = Form(...),
    lat: float = Form(...),
    lon: float = Form(...),
    start_date: str = Form(...),
    modalities: list[str] = Form(...),
    days: int = Form(...)
):
    tier, limits = validate_user_request(api_key, modalities, days)

    result = generate_prediction_datacube(
        lat=lat,
        lon=lon,
        start_date_str=start_date,
        user_tier=tier,
        modalities=modalities[:len(limits["modalities"])],  # enforce
        days=min(days, limits["days"]),
        max_threads=limits["threads"]
    )

    return {
        "tier_used": tier,
        "result": result
    }


In [ ]:
def generate_prediction_datacube(lat, lon, start_date_str, user_tier, modalities, days, max_threads):
    # generates  data cubes for HAB using only the modalities, days, and thread resources allowed by the user’s tier, ensuring resource control and secure model execution.
    ...
    with ThreadPoolExecutor(max_workers=max_threads) as executor:
        ...


In [ ]:
@app.get("/me/limits")
def get_user_limits(api_key: str):
    user = get_user_by_api_key(api_key)
    if not user:
        raise HTTPException(403, "Invalid API key.")
    tier = user["tier"]
    limits = TIER_LIMITS[tier]
    return {
        "tier": tier,
        "limits": limits,
        "usage": user["usage_count"]
    }
# access limits based on their API key,